# 04 — Dataset, Modell, Backtest & Trade Journal (Phase 2b)

**Voraussetzung:** Notebooks 02 und 03 sind gelaufen — Kurse, Makro und (optional) gescorte News liegen in `data/`.

Was hier passiert:
1. Supervised Dataset bauen (TA + Sentiment + Makro → Forward-Return-Label)
2. XGBoost-Modell trainieren (klassifiziert Down/Flat/Up)
3. Holdout-Evaluation
4. Walk-Forward Backtest
5. Trade Journal Demo
6. SHAP-Attribution für eine Beispiel-Empfehlung

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data import universe
from src.features import build_dataset
from src.model import backtest as bt_mod
from src.model import journal as journal_mod
from src.model.predictor import Predictor
from src.model.journal import PredictionRecord

sns.set_style('whitegrid')

## 1. Dataset bauen (ETFs als Test-Universum)

Wir starten mit den 30 ETFs — schnell zu trainieren, weniger Noise als Einzelaktien.
Setze `symbols = universe.symbols()` für alle 500 S&P 500.

In [ ]:
symbols = universe.etf_symbols()
print(f'Symbols: {len(symbols)}')

dataset = build_dataset.build_dataset(symbols, horizon_days=5, up_thresh=0.01, down_thresh=0.01)
print(f'Dataset shape: {dataset.shape}')
print(f'Klassenverteilung:\n{dataset["fwd_class"].value_counts(normalize=True).rename({-1:"down", 0:"flat", 1:"up"})}')

In [ ]:
feature_cols = build_dataset.feature_columns(dataset)
print(f'Anzahl Features: {len(feature_cols)}')
feature_cols

## 2. Train / Validation / Test Split (chronologisch!)

**Niemals** random splits bei Zeitreihen — sonst leakt Zukunftsinfo in den Trainingsset.
Aufteilung: erste 80% Train, nächste 10% Val (Early Stopping), letzte 10% Test.

In [ ]:
dataset = dataset.sort_values('date').reset_index(drop=True)
n = len(dataset)
i_train = int(n * 0.8)
i_val = int(n * 0.9)
train_df = dataset.iloc[:i_train]
val_df   = dataset.iloc[i_train:i_val]
test_df  = dataset.iloc[i_val:]
for name, d in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name:6} {len(d):>6} rows  {d["date"].min().date()} -> {d["date"].max().date()}')

## 3. Modell trainieren

Falls `device='cuda'` in den Default-Params nicht greift (kein PyTorch/CUDA), schaltet XGBoost automatisch auf CPU.

In [ ]:
predictor = Predictor(feature_cols=feature_cols, horizon_days=5)
predictor.train(train_df, df_val=val_df)
print('Training fertig.')
predictor.save('predictor')

## 4. Holdout-Evaluation

In [ ]:
metrics = predictor.evaluate(test_df)
print(f'Test rows: {metrics["n"]}')
print('\nClassification Report:')
rep = pd.DataFrame(metrics['report']).T
display(rep.round(3))

print('\nConfusion Matrix (rows=true, cols=pred), [-1, 0, 1]:')
cm = pd.DataFrame(metrics['confusion'], index=['true_-1','true_0','true_1'], columns=['pred_-1','pred_0','pred_1'])
display(cm)

In [ ]:
test_scores = predictor.predict_score(test_df)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(test_scores, bins=50, alpha=0.7)
ax.set_title('Verteilung der Test-Scores  (P(up) - P(down))')
ax.set_xlabel('Signed Score')
plt.show()

## 5. Walk-Forward Backtest

**Ehrliche** Performance-Schätzung: für jede Test-Periode wird das Modell nur auf Daten *davor* trainiert.
Dauert je nach Universum 1–10 Minuten.

In [ ]:
config_bt = bt_mod.BacktestConfig(
    horizon_days=5,
    rebalance_days=5,
    top_n_long=5,        # mit ETF-Universum (~30) sind Top-5 sinnvoll
    top_n_short=0,       # erstmal long-only
    cost_bps=5.0,
    initial_capital=100_000.0,
)

def factory():
    return Predictor(feature_cols=feature_cols, horizon_days=5)

result = bt_mod.walk_forward(dataset, factory, config_bt)
print('=== Backtest-Metriken ===')
for k, v in result.metrics.items():
    print(f'  {k:24} {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
result.equity_curve.plot(ax=ax, color='steelblue', linewidth=1.5, label='Strategie')
ax.axhline(config_bt.initial_capital, color='gray', linestyle='--', alpha=0.5, label='Startkapital')
ax.set_title(f'Equity Curve  (Sharpe {result.metrics["sharpe"]:.2f}, MDD {result.metrics["max_drawdown"]*100:.1f}%)')
ax.set_ylabel('EUR')
ax.legend()
plt.show()

In [ ]:
result.trades.head(20)

## 6. Trade Journal — Demo

Jede Empfehlung kommt mit allen Inputs in die SQLite-DB. Nach Ablauf der Halteperiode wird Outcome ergänzt.
Das ist die Grundlage für Phase 3 Postmortems.

In [ ]:
# Top-Empfehlung des heutigen Tags loggen
today = test_df.sort_values('date').groupby('symbol').tail(1)
today = today.assign(score=predictor.predict_score(today))
top = today.nlargest(1, 'score').iloc[0]

rec = PredictionRecord(
    asof_date=str(top['date'].date()),
    symbol=top['symbol'],
    score=float(top['score']),
    action='long',
    horizon_days=5,
    top_features={'rsi_14': float(top['rsi_14']), 'sent_mean': float(top.get('sent_mean', 0))},
    sentiment_inputs={'n_articles': int(top.get('n_articles', 0))},
    macro_snapshot={'vix': float(top.get('vix', 0))},
)
pred_id = journal_mod.log_prediction(rec)
print(f'Logged prediction id: {pred_id}')

# Outcome simulieren (in Live-Modus macht das der daily-Orchestrator)
journal_mod.log_outcome(pred_id, realised_return=float(top['fwd_return']),
                        success=top['fwd_return'] > 0)

print('\nJournal Summary:')
for k, v in journal_mod.journal_summary().items():
    print(f'  {k:24} {v}')

## 7. SHAP-Attribution für eine Empfehlung

Zeigt welche Features wie viel zum Score beigetragen haben.

In [ ]:
from src.explain.shap_attribution import ShapExplainer

expl = ShapExplainer(predictor)
top_row = top.to_frame().T.reset_index(drop=True)
attribution = expl.top_features_for_row(top_row, row_idx=0, top_n=10)
display(attribution)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if s > 0 else 'red' for s in attribution['shap']]
ax.barh(attribution['feature'], attribution['shap'], color=colors)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title(f'SHAP-Attribution für {top["symbol"]} (Score {top["score"]:+.2f})')
ax.set_xlabel('Beitrag zum Score')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Nächste Schritte

Wenn dieses Notebook sauber durchläuft ist die Modell-Pipeline einsatzbereit. Dann:

- **`python scripts/run_daily.py`** — täglicher EOD-Lauf
- **`python scripts/run_premarket.py`** — Pre-Market-Anpassung
- **`python scripts/run_backtest.py`** — Backtest mit verschiedenen Parametern
- **`streamlit run app/dashboard.py`** — Dashboard
- **`python -m app.telegram_bot`** — Telegram-Bot
- **`python scripts/run_postmortems.py`** — LLM-Analyse der Fehltrades
- **`python scripts/ingest_edgar.py --universe sp500`** — RAG-Wissensdatenbank befüllen